# GMRES

GMRES is a method of solving linear systems of equations $A_{ij} x_j = b_i$ ($i,j=1,2,...,N$). 

The idea is construct a (Krylov) vector space of dimension $m$

$$
\mathcal{K}_m = span\left[b, A \cdot b , A^2 \cdot b,  \dots, A^{n-1} \cdot b, A^m \cdot b  \right]
$$

and look for a solution in $\mathcal{K}_m$. If $m=N$, the exact solution ($x$) can be found in $\mathcal{K}_N$. In many cases, we are interested in looking inside $\mathcal{K}_{m \ll N}$ for an approximate solution.

This notebook explain the basic idea and algorithm behind GMRES, a method of buildig an orthonormal basis for $\mathcal{K}_m$ and iteratively looking for approximate solution to $A_{ij} x_j = b_i$.

# Arnoldi or "Gram–Schmidt orthonormalization of Kylov subspace". 


The Arnoldi loop builds $m+1$ basis vectors, $v^{(1)}$, $v^{(2)}$ ,..., $v^{(m+1)}$.

During Arnoldi, we also fill the "Hessenberg" matrix $h_{ij}$ with $i=1,2,...,m+1$ and $j=1,2,...,m$. This matrix holds the projection of $A$ the basis vectors $v$ because if comes from the "Gram-Schmidt"-like Arnoldi loop:

---

Start with  a guess solution, $x^{0}$, and find the first vector

$$v^{(1)}_i= (b_i - A_{ij} x^{0}_j)/\beta$$

with $\beta = |(b_i - A_{ij} x^{0}_j)|$.

For each k=1,2,...,m, we construct the next basis vectors ($v^{(k+1)}$) starting from $w_{f}^{(k+1)}$:

- $w_{f}^{(k+1)} \leftarrow  \sum_{l=1}^{N} A_{fl} v^{(k)}_{l}$ (for all f=1,2,...,N)

Then, we subtract the projections of all previous basis vectors ($v^{(1)}$, $v^{(2)}$,..., $v^{(k)}$) from $w_{f}^{(k+1)}$.

- For i=1,2,...,k
- - $h_{ik} \leftarrow \sum_{f=1}^{N}  v^{(i)}_{f} w_{f}^{(k+1)}$

- - $w_{f}^{(k+1)} \leftarrow  w_{f}^{(k+1)} - h_{ik} \, v^{(i)}_{f} $ (f=1,2,...,N)

<p style="text-align: right;">
This loop basically constructs  $ T_{f} = w_{f}^{(k+1)} - \sum_{i=1}^{k} v^{(i)}_{f} \sum_{l=1}^{N}  v^{(i)}_l w_{l}^{(k+1)} $, which obeys (for $p<k+1$)
</p>
<p style="text-align: right;">
$
 \sum_{f=1}^{N} v^{(p)}_{f} T_{f}   = \sum_{f=1}^{N} v^{(p)}_{f} w_{f}^{(k+1)} - \sum_{f=1}^{N} v^{(p)}_{f} \sum_{i=1}^{k} v^{(i)}_{f} \sum_{l=1}^{N}  v^{(i)}_l w_{l}^{(k+1)}= 
$
</p>

<p style="text-align: right;">
$
\sum_{f=1}^{N} v^{(p)}_{f} w_{f}^{(k+1)} - \sum_{i=1}^{k} \delta_{ip} \sum_{l=1}^{N}  v^{(i)}_l w_{l}^{(k+1)} =
$ 
</p>
<p style="text-align: right;">
$
\sum_{f=1}^{N} v^{(p)}_{f} w_{f}^{(k+1)} -  \sum_{l=1}^{N}  v^{(p)}_l w_{l}^{(k+1)} = 0
$
</p>

Then, we normalize  $w_{f}^{(k+1)}$ using

- $h_{k+1 \, k} \leftarrow |\vec{w}^{(k+1)}|$ 
- $v_{f}^{(k+1)} \leftarrow w_{f}^{(k+1)}/h_{k+1 \, k}$ (f=1,2,...,N)

---

In this loop, we basically built:

for k=1,2,...,m:
- $h_{1k}$, $h_{2k}$,..., $h_{kk}$, $h_{k+1\,k}$.
- By construction, $h_{i\,k}=0$ for $i>k+1$.
- $v^{(k+1)}$


As mentioned before the start of the loop, this allows us to decompose

$$
\sum_{l=1}^{N} A_{fl} v^{(k)}_{l}=v_{f}^{(k+1)} h_{k+1\,k} + \sum_{i=1}^k v_{f}^{(i)} h_{ik} = \sum_{i=1}^{k+1} v_{f}^{(i)} h_{ik}
$$

Note that this is true during the iteration because each new basis vector is orthogonalized with the previous ones.

---

## Subdle point

$A \cdot v^{(k)}$ constructs vector $v^{(k+1)}$. So, although the subspace has dimension $m$, we construct $m+1$ orthonormal vectors. The reason is that the first basis vector also takes $b$ into account. This is the nature of the Arnoldi loop. In any case, the relation that matters at the end is

$$
\sum_{l=1}^{N} A_{fl} v^{(k)}_{l}= \sum_{i=1}^{k+1} v_{f}^{(i)} h_{ik},
\, {\rm for \,\, all} \,\,\, k=1,2,...,m \;.
$$

That is, if we want to find a solution in $\mathcal{K}_{m}$,  $x_{f} = c_1 \, v^{(1)}_{f} + c_2 \, v^{(2)}_{f} \dots c_m \, v^{(m)}_{f}$, we need to know $ v^{(m+1)}_{f}$ because 

$$ 
\sum_{i=1}^N A_{fi} x_i =  \sum_{i=1}^N A_{fi} \sum_{k}(c_k \, v^{(k)}_{i}) = 
\dots c_1 \sum_{i=1}^N A_{fi} \, v^{(1)}_{i} + c_2 \sum_{i=1}^N A_{fi}  \, v^{(2)}_{i} +
\dots + c_m \sum_{i=1}^N A_{fi} \, v^{(m)}_{i} = \dots \sum_{i=1}^{m+1} v_{f}^{(i)} h_{im} \,.
$$

That is, when $A$ acts on the $m^{th}$ component of $x$, it produces a projecton to $v^{(m+1)}$. This is because the $\mathcal{K}_{m}$ does not cover the entire $N$-dimension vector space that $A$ defines.  If we choose $m=N$, then $h_{m+1 \, m} = 0$. So, $h_{m+1 \, m}$ also help us see ho well we cover the full dimension $N$ vector space.




# Looking for a solution

Once we have the subspace basis, we look for a solution of the form

$$
x_{f} = x_{f}^{(0)} + \sum_{r=1}^{m} v^{(r)}_{f} \, y_{r}.
$$

Given a vector $y$, the residual vector is defined as

$$
r_{f}(y)=b_{f}−\sum_{l=1}^N A_{fl} x_l = b_{f}−\sum_{l=1}^N A_{fl}
\left( x_l^{(0)} + \sum_{r=1}^{m} v^{(r)}_{l} \, y_{r} \right)=
 \sum_{k=1}^{m+1} v^{(k)}_{f} \left(\beta \delta_{k1} −  \sum_{r=1}^{m} h_{kr} \, y_{r}\right)
$$

An approximate solution may be found by minimizing $|r(y)|$ with respect to $y$. Since all vectors $v$ are orthonormal, 

$$
|r(y)|^2 = \sum_{k=1}^{m+1} \left(\beta \delta_{k1} −  \sum_{r=1}^{m} h_{kr} \, y_{r}\right)^2 \;.
$$

--

## Observation

- Triangular systems are easy to solve.

- $h_{ik}$ is almost triangular because $h_{i \, k} =0$ for $i>k+1$.

- Orthogonal rotations do not change the value of $|r(y)|$.

- We only need to rotate ($h \to \tilde h$) for $i=k+1$ and make $\tilde h_{k+1\,k}=0$.


So, let's perform an orthogonal transformation on $h_{i \, k}$ to make it triangular.  

# Givens rotations


The way we do it is simple. Start with  the vector 
$$T_i = g_{i} −  \sum_{j=1}^{k} h_{ij} \, y_{j}$$   
with $g_i=\beta \delta_{i1}$.

Perform orthogonal transformation $\hat O$ (sum over i is implied)

$$O_{li}T_{i} = O_{li}g_{i} −  \sum_{j=1}^{k} (O_{li}h_{ij}) \, y_{j} $$

The goal is to make $\tilde h = \hat O \, h$ upper triangular. By construction, $h_{ik}=0$ for $i>k+1$. So, we only need to rotate for $i=k+1$ and make $\tilde h_{k+1\,k}=0$. 

To do this during the Arnoldi loop, we note that we can decompise $\hat O = \hat O^{(k)} \, \hat O^{(k-1)} \,... \hat O^{(1)}$, with each $\hat O^{(p)}$ a rotation of the plane $p$ and $p+1$. This means that  

$$
\hat O^{(p)}_{ij}=\left( \begin{matrix} 
                        c_p & s_p \\ 
                        -s_p & c_p 
                    \end{matrix}\right), \quad {\rm for}\, (i,j)=(p,p+1)
$$

and for all other values of $i$ and $j$,
$$
O^{(p)}_{ij} = \delta_{ij} 
$$

Then,  during the $k^{th}$ step of Arnoldi loop, we compute

\begin{eqnarray}
&\tilde h_{k\,k} = O^{(k)}_{kj}h_{jk}= c_k \, h_{kk} + s_k \, h_{k+1\,k} = r
\\
&\tilde h_{k+1\,k} = O^{(k)}_{k+1\,j}h_{jk}= -s_k \, h_{kk} + c_k \, h_{k+1\,k} = 0
\end{eqnarray}

This means that $c_k=h_{kk}/r$, $s_k=h_{k+1\,k}/r$, and $r^2=h^2_{k\,k}+h^2_{k+1\,k}$ (so that $c_k^2+s_k^2=1$).

## A subdle point.

At the $k^{th}$ step of the Arnoldi loop, you create $h_{k,k}$, $h_{k+1,k}$, and also all $h_{i,k}$ with $i=1,2,...,k-1$. This means that you need to apply $\hat O^{(k-1)}$ to those, to keep the transportation fully orthodonal. That is, every step, after we create $h_{1\,k}$, $h_{2\,k}$,..., $h_{k-1\,k}$, $h_{k\,k}$, $h_{k+1\,k}$, we apply the rotation 
$$
O_{li}h_{ik}=O^{(k)}_{l\,i_k} O^{(k-1)}_{i_{k}\,i_{k-1}}...O^{(2)}_{i_3\,i_2} O^{(1)}_{i_2\,i_1} h_{i_{1}\, k} 
$$

this gives us the "algorithm":

At step $k$:
- for i in 1,2,...,k-1

- - compute $\tilde h_{lk} = O^{(i)}_{l\,j} h_{j\, k}$. Here the transformation only applies for $l=i,i+1$  (all others indices result in multiplication with $\delta_{li}$). So, I compute

\begin{eqnarray}
&\tilde h_{i\,k} = O^{(i)}_{ij}h_{jk}= c_i \, h_{ik} + s_i \, h_{i+1\,k}  
\\
&\tilde h_{i+1\,k} = O^{(i)}_{i+1\,j}h_{jk}= -s_i \, h_{ik} + c_i \, h_{i+1\,k}  
\end{eqnarray}

Once this loop exits, we have transformed all $h_{ik}$ up to $h_{k-1\,k}$. For the $k^{th}$ component, we impose

\begin{eqnarray}
&\tilde h_{k\,k} = O^{(k)}_{kj}h_{jk}= c_k \, h_{kk} + s_k \, h_{k+1\,k} = r
\\
&\tilde h_{k+1\,k} = O^{(k)}_{k+1\,j}h_{jk}= -s_k \, h_{kk} + c_k \, h_{k+1\,k} = 0
\end{eqnarray}

This gives us the values of $O^{(k)}$, which will be used in the next iteration duing the "$i$" loop.

To keep the transformation of $T_i$ consistent, we also transform (at every step $k$) $g_k$ with

$$
\tilde g_k = O^{(k)}_{ki}g_{i},\qquad
\tilde g_{k+1} = O^{(k)}_{k+1i}g_{i}
$$


# Minimization

We can finally find y that minimizes (using $h_{ij}=0$ for $j<i$)

$$
 \sum_{i=1}^{k+1} \left(g_i −  \sum_{j=1}^{k} h_{ij} \, y_{j} \right)^2 =
 \sum_{i=1}^{k+1} \left(g_i −  \sum_{j=i}^{k} h_{ij} \, y_{j} \right)^2 =
 \sum_{i=1}^{k} \left(g_i −  \sum_{j=i}^{k} h_{ij} \, y_{j} \right)^2 + g_{k+1}^2
$$

The last term is there because for $i=k+1$, $\sum_{j=i}^{k} h_{ij} \, y_{j}=0$.

Since each term is positive, minimization wrt to $y$ means minimization of $\sum_{i=1}^{k} \left(g_i −  \sum_{j=i}^{k} h_{ij} \, y_{j} \right)^2$. There are enough free components to make this $0$, so we just solve

$$
\sum_{j=i}^{k} h_{ij} \, y_{j}=g_i, \quad {\rm for} \,\, i=1,2,...,k 
$$

Since $h$ is triangular, we can solve it using "backsubstitution". Doing this, the minimimum of $|r|$ becomes (up to roundoff errors)

$$|r| = |g_{k+1}|$$

# Solving triangular systems -- backsubstitution
If we have a system 

$$M_{ij}y_j=c_i$$

with $i,j = 1,2,...,k$ and $M_{ij}$ upper triangular ($M_{ij}=0$ for $i>j$), the system can be solved using backsubstitution.

First we solve the $k^{th}$ equation -- because only $M_{kk}\neq 0$ -- as

$y_k=c_k/M_{kk}$

Since we have $c_k$,  we can solve  

$$ M_{k-1,k-1}y_{k-1}+M_{k-1,k}y_{k}=c_{k-1} =>  y_{k-1} = (c_{k-1} - M_{k-1,k}y_{k})/M_{k-1,k-1} $$

And continue. This is called backsubstitution, with general formula

$$y_k=c_k/M_{kk}$$

$$y_{k-i} = \dfrac{c_{k-i} - \sum_{j=k-i+1}^{k}  M_{k-i,j}y_{j}}{M_{k-i,k-i}}$$

# Restarting

Once we compute $y$, we can find a new $x$

$$
x_{f} = x_{f}^{(0)} + \sum_{r=1}^{m} v^{(r)}_{f} \, y_{r}.
$$

This can be used as $x_{f}^{(0)}$ for a new Arnoldi loop. This is called restart. We use it when $m<N$, because GMRES may need to look at many different Krylov subspaces to find a good solution. 



---

#### Now we have all the ingredients. We only have to put the givens rotations inside the Arnoldi loop and solve the triangular system once the Arnoldi loop finishes.

---

Basic mathematical functions

In [1]:
import numpy as np # only for zeros, random, sums of arrays and things like that.

In [2]:
#This defines the product between matrix and vector. GMRES only need vectors A.v, so this will be the only thing I use.
def matvec(A,v):
    N=len(A)
    c=np.zeros(N)
    for i in range(N):
        for j in range(N):
            c[i]+=A[i][j]*v[j]
    return c    

#I will take iterative steps to find a small enough residual 
def residual(A,b,test_solution):
    return b - matvec(A,test_solution)

#I will need also the norm of vectors
def norm(v):
    n=0
    for i in range(len(v)):
        n+=v[i]**2.
    return n**0.5

def vector_scale(v,a):
    N=len(v)
    v_div_a=np.zeros(N)
    for i in range(N):
        v_div_a[i]=v[i]/a
    return v_div_a

def dot(v,u):
    N=len(v)
    prod=0
    for i in range(N):
        prod+=v[i]*u[i]
    return prod      

In [3]:
# backsubstitution
def triangular_solve(R,d,k):
    #I take k, in case I want to solve a smaller system
    
    y=np.zeros(k)
    for i in range(1,k+1):
        s=0
        for j in range(k-i+1,k):
            s+=R[k-i][j]*y[j]
        y[k-i]=(d[k-i]-s)/R[k-i][k-i]

    return y

In [4]:
#test triangular_solve

if False:
    N=45
    R=np.zeros([N,N])
    d=np.zeros(N)
    for i in range(N):
        d[i]=np.random.uniform(-1,1)
        for j in range(i+1):
            R[j][i]=np.random.uniform(-1,1)
    
    for i in range(N):
        R[i][i]+=10

    #any dimension smaller than N should work
    x=np.array(triangular_solve(R,d,N-3))
    
    
    print(norm(R[:N-3,:N-3]@x-d[:N-3])/norm(d[:N-3]))

In [5]:
# setup the problem again and rewrite the Arnoldi loop
#setup the problem
N=80
subspace_dimension=25#choose it to be smaller than N. This is what makes GMRES faster than LU or other algorithms

#symetric systems can be solved
A=np.random.uniform(0,5,[N,N])
A=A+A.T

#nearly diagonal systems can be solved
# A=np.eye(N,N)+7e-2*np.random.uniform(-5,5,[N,N])

rhs=np.random.uniform(-1,1,N)


v=np.zeros([subspace_dimension+1,N])

h=np.zeros([subspace_dimension+1,subspace_dimension])

x=np.zeros_like(rhs)

#recursively update x
for iteration in range(1500):
    
    v=np.zeros([subspace_dimension+1,N])
    h=np.zeros([subspace_dimension+1,subspace_dimension])

    #compute the residual
    res=residual(A,rhs,x)
    beta=norm(res)
    
    #first basis vector
    v[0] = vector_scale(res,beta) 
    
    e1=np.zeros(subspace_dimension+1)
    e1[0]=1
    g=beta*e1
    c=np.zeros(subspace_dimension+1)
    s=np.zeros(subspace_dimension+1)
    
    # for testing I keep the original h and g.
    h_original=np.zeros([subspace_dimension+1,subspace_dimension])
    g_original=g[:]
    
    for k in range(subspace_dimension):
        #new vector (this will be the next basis vector after orthonormalization)
        w = matvec(A, v[k])
    
        #orthogonalize v_0, v_1, ... v_j
        for i in range(k + 1):
            
            h[i][k] = dot(v[i], w)
            w -= h[i][k] * v[i]
            #only for testing
            h_original[i][k]=h[i][k]
    
        h[k+1][k] = norm(w)
        h_original[k+1][k]=h[k+1][k]
    
        #I will use this later to see if I need to break the loop
        test_h=h[k+1][k]
        
    
        # normalize next basis vector (if h[k + 1][k]=0, just put v[k + 1]=0)
        if test_h<1e-10:
            pass
            #v[k+1] will remain equal to 0 (from its initialization)
        else:    
            v[k+1] = vector_scale(w, h[k+1][k])  # if vector_scale = divide by scalar
    
        #here, I can apply rotations to h[i][k] and h[k + 1][k]
    
        # first, I apply O^{(i)} i=0,1,...,k-1 (range(k) generates this)
        for i in range(k):
            p1=h[i][k]
            p2=h[i+1][k]
            h[i][k]=c[i]*p1+s[i]*p2
            h[i+1][k]=-s[i]*p1+c[i]*p2
            
            
        
        p1=h[k][k]
        p2=h[k+1][k]
        r=np.sqrt(p1**2+p2**2)
        if r<1e-10:
            k_end=k
            break
            
            c[k]=1
            s[k]=0
        else:
            c[k]=p1/r
            s[k]=p2/r
        h[k][k]=r
        h[k+1][k]=0 # I put 0 explicitely because -s[k]*a+c[k]*b can have roundoff errors
    
        
        # i need to apply the transformation to g as well
        g0=c[k]*g[k]+s[k]*g[k+1]
        g1=-s[k]*g[k]+c[k]*g[k+1]
        g[k]=g0
        g[k+1]=g1

        
        #the biggest k_end is subspace_dimension because the k loop run up to subspace_dimension-1.
        #if the loop exits earlier, I will treat k_end as the effective subspace_dimension.
        # so, whatever would run up to subspace_dimension after the Arnoldi loop, I will run
        # up to k_end
        k_end=k+1#this will tell me at which k we exit the loop(in case we exit earlier)
        
        #if the norm of h[k + 1][k] (before transformation) vanishes, we have exhausted the linearly independent
        #vectors we can have from the guess x0. Maybe we need to restart
        #with a different guess (if solution cannot be found).
        if test_h < 1e-10:
            break
    

    # Solve the active triangular system only
    # y = triangular_solve(h, g, k_end)#instead of subspace_dimension use k_end in case the loop finished earlier.

    #notice that first we fill y[k_end-1], then y[k_end-2], etc.
    #So, k_end=k+1 is consistent with the convension (no y[k_end] is generated).
    y=np.zeros(k_end)
    for i in range(1,k_end+1):
        s=0
        for j in range(k_end-i+1,k_end):
            s+=h[k_end-i][j]*y[j]
        y[k_end-i]=(g[k_end-i]-s)/h[k_end-i][k_end-i]

    
    dx = np.zeros_like(x)
    for j in range(k_end):#instead of subspace_dimension use k_end in case the loop finished earlier.
        dx += y[j] * v[j]
    x += dx

    
    print(norm(A@x-rhs),norm(rhs),end='\t')
    print(norm(g[:k_end-1]),np.abs(g[k_end]))
    # print(np.abs(g[k_end]))#this shoould be close to norm(A@x-rhs)
    if norm(A@x-rhs)/norm(rhs) < 1e-3:
        break
norm(A@x-rhs),norm(rhs)

0.7408155317213588 4.952475771989374	4.885619496583814 0.7408155317213606
0.38059907177812635 4.952475771989374	0.6321311574472152 0.3805990717781267
0.29235628564776306 4.952475771989374	0.23653350850713664 0.29235628564776217
0.2454231184135881 4.952475771989374	0.15756265926376775 0.24542311841358846
0.21758755736695 4.952475771989374	0.11052277906719166 0.21758755736694957
0.19798040314610754 4.952475771989374	0.08921934303031077 0.197980403146108
0.18328530483542932 4.952475771989374	0.07295925077257741 0.18328530483542924
0.1719747349598412 4.952475771989374	0.06257479216278983 0.17197473495984106
0.16312104997635476 4.952475771989374	0.0531384451667564 0.1631210499763535
0.1560987945091457 4.952475771989374	0.04669209653832485 0.15609879450914504
0.15040348390326921 4.952475771989374	0.04072235521780384 0.15040348390326846
0.14570126059080885 4.952475771989374	0.03675947912311943 0.14570126059080746
0.14171394522494885 4.952475771989374	0.03292714982607892 0.1417139452249487
0.1

(np.float64(0.004941026060396091), np.float64(4.952475771989374))

In [6]:
for i in range(k_end):
    # check that h is upper triangular
    a=np.max(np.abs(h[i+1:,i]))
    if a>0:
        print(i,a)

for any vector $y$ , $T_{i}=\left\{ \sum_{i=1}^{k+1} \left(g_i −  \sum_{j=1}^{k} h_{ij} \, y_{j} \right)^2 \right\}$ should be te same both for the transformed and the original $h$ and $g$.

In [7]:
for test_loop in range(500):
    #check that |T| is the same both with h,g and with h_original, g_original
    T=np.zeros(k_end+1)
    T_original=np.zeros(k_end+1)
    
    y=np.random.uniform(-5,5,k_end)
    
    for i,_ in enumerate(T):
        s=0
        s_original=0
        for j in range(k_end):
            s+=h[i][j]*y[j]
            s_original+=h_original[i][j]*y[j]
        T[i]=g[i]-s
        T_original[i]=g_original[i]-s_original
    
    if np.abs(norm(T)/norm(T_original)-1)>1e-2:
        print(y, np.abs(norm(T)/norm(T_original)-1)*100)